# VIIRS-Hotspots filtern
### Von einer Wärmeanomalie zum Vegetations-Hotspot

Ein VIIRS-Hotspot zeigt eine auffällige Wärmequelle. Liegt sein Mittelpunkt auf
Vegetation? Dieses Notebook beantwortet diese Frage mit **ESA WorldCover 2021**
und sammelt die passenden Punkte in einem **Hosted Feature Layer in ArcGIS Online**.

**VIIRS laden → bekannte Punkte entfernen → Vegetation filtern → Land ergänzen → speichern**

Der Ablauf ist bewusst einfach: Wir behalten die Confidence-Klassen `nominal`
und `high` und lesen pro Hotspot genau einen WorldCover-Pixel am Mittelpunkt aus.
Das Ergebnis sind mögliche Vegetationsbrände, keine bestätigten Brandereignisse.

**Start:** In ArcGIS Online mit der **Advanced-Notebook-Runtime** öffnen und die
Zellen von oben nach unten ausführen. Benötigt werden ArcPy/Spatial Analyst,
Zugriff auf die drei Datenquellen und das Recht, Hosted Feature Layer zu erstellen
und zu bearbeiten. Die Anmeldung erfolgt über `GIS("home")`.
[Hinweise zur Runtime](https://doc.arcgis.com/en/arcgis-online/reference/use-arcpy-in-your-notebook.htm).

Beim ersten Lauf wird der Ziellayer angelegt, danach wird derselbe Layer ergänzt.
Jeder vollständige Lauf schreibt Daten und entfernt Punkte, die älter als sieben Tage sind.

## 1 · Die Filter einstellen

Die wichtigsten Stellschrauben stehen hier. `ROLLING_HOURS` bestimmt, wie lange
Punkte im Ergebnis bleiben. `SOURCE_OVERLAP_HOURS` erweitert den neuesten
Altersblock der Quelle: Ist der jüngste Punkt beispielsweise drei Stunden alt,
laden wir bei einer Stunde Überlappung alle Punkte mit `hours_old <= 4`.

`VEGETATION_CODES` enthält die erlaubten Landbedeckungen:

| Code | Klasse | Code | Klasse |
|---:|---|---:|---|
| 10 | Bäume | 40 | Ackerland |
| 20 | Sträucher | 90 | Krautige Feuchtgebiete |
| 30 | Grasland | 95 | Mangroven |
| 100 | Moose und Flechten | | |

Gebäude, Wasser, Schnee/Eis, vegetationsarme Flächen und fehlende Rasterwerte
werden ausgeschlossen. Die Einstellungen gelten für neu verarbeitete Punkte.

In [ ]:
# Hier die Filter für den eigenen Anwendungsfall anpassen.
ROLLING_HOURS = 168                         # 7 Tage im Ziellayer behalten
SOURCE_OVERLAP_HOURS = 1                    # Überlappung beim stündlichen Abruf
CONFIDENCE_VALUES = ["nominal", "high"]       # "low" wird ausgeschlossen
VEGETATION_CODES = {10, 20, 30, 40, 90, 95, 100}

# Name und eindeutiges Such-Tag des Ziellayers; für einen neuen Showcase beide ändern.
SERVICE_NAME = "viirs_vegetation_hotspots_notebook"
SERVICE_TITLE = "VIIRS – Vegetations-Hotspots"
SERVICE_TAG = "viirs-hourly-notebook-layer"

### Datenquellen und Verbindung

VIIRS und WorldCover verwenden die bisherigen Quellen. Für die Länderzuordnung
nutzen wir eine öffentlich abfragbare Kopie von Natural Earth im Maßstab
1:50 Millionen. Der bisherige Länderlayer auf `demoportal12.esri.de` verlangt
eine zusätzliche Anmeldung. Bei Bedarf unter `COUNTRIES_URL` einen eigenen
erreichbaren Länderlayer mit dem Feld `SOVEREIGNT` eintragen.

In [ ]:
from datetime import datetime, timedelta, timezone

import arcpy
import pandas as pd
from IPython.display import Markdown, display
from arcgis.features import FeatureLayer, FeatureLayerCollection, GeoAccessor
from arcgis.gis import GIS

VIIRS_URL = (
    "https://services9.arcgis.com/RHVPKKiFTONKtxq3/arcgis/rest/services/"
    "Satellite_VIIRS_Thermal_Hotspots_and_Fire_Activity/FeatureServer/0"
)
WORLDCOVER_URL = (
    "https://tiledimageservices.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/"
    "European_Space_Agency_WorldCover_2021_Land_Cover_WGS84_7/ImageServer"
)
COUNTRIES_URL = (
    "https://services.arcgis.com/0xnwbwUttaTjns4i/ArcGIS/rest/services/"
    "Natural_Earth_quick_start/FeatureServer/43"
)

# Die englischen Klassenwerte bleiben mit dem bisherigen Ziellayer kompatibel.
WORLDCOVER_CLASS = {
    10: "Tree cover", 20: "Shrubland", 30: "Grassland", 40: "Cropland",
    50: "Built-up", 60: "Bare or sparse vegetation", 70: "Snow and ice",
    80: "Permanent water bodies", 90: "Herbaceous wetland",
    95: "Mangroves", 100: "Moss and lichen",
}

gis = GIS("home")
source = FeatureLayer(VIIRS_URL, gis)
countries = FeatureLayer(COUNTRIES_URL, gis)
arcpy.env.overwriteOutput = True

# Ein gemeinsamer UTC-Zeitpunkt für diesen Lauf.
now = datetime.now(timezone.utc)
now_ms = int(now.timestamp() * 1000)
cutoff = now - timedelta(hours=ROLLING_HOURS)
cutoff_sql = f"TIMESTAMP '{cutoff:%Y-%m-%d %H:%M:%S}'"

## 2 · Den Ziellayer vorbereiten

Das Notebook sucht im eigenen Konto nach `SERVICE_TAG`. Gibt es noch keinen
Treffer, legt es einen Punktlayer an. Pro Showcase genau dieses eine Tag verwenden.
Die Feldnamen bleiben gleich, damit vorhandene Karten und Dashboards weiter darauf
zugreifen können. `detection_id` erkennt dieselbe Beobachtung bei späteren Läufen wieder.

Der folgende Block ist die Einrichtung; die eigentliche Filterung beginnt in Schritt 3.

In [ ]:
items = gis.content.search(
    query=f'tags:"{SERVICE_TAG}" AND owner:{gis.users.me.username}',
    item_type="Feature Layer",
    max_items=2,
)
if len(items) > 1:
    raise RuntimeError("Das SERVICE_TAG muss genau einen Ziellayer bezeichnen.")

if items:
    item = items[0]
else:
    item = gis.content.create_service(
        name=SERVICE_NAME,
        service_type="featureService",
        has_static_data=False,
        capabilities="Query,Create,Update,Delete,Editing",
        wkid=4326,
    )
    definition = {
        "name": "VIIRS Vegetations-Hotspots",
        "type": "Feature Layer",
        "geometryType": "esriGeometryPoint",
        "objectIdField": "OBJECTID",
        "fields": [
            {"name": "OBJECTID", "type": "esriFieldTypeOID", "alias": "OBJECTID", "nullable": False, "editable": False},
            {"name": "detection_id", "type": "esriFieldTypeString", "alias": "Beobachtungs-ID", "length": 100, "nullable": False},
            {"name": "acq_time", "type": "esriFieldTypeDate", "alias": "Aufnahmezeit (UTC)", "nullable": False},
            {"name": "hours_old", "type": "esriFieldTypeDouble", "alias": "Alter in Stunden"},
            {"name": "frp", "type": "esriFieldTypeDouble", "alias": "Strahlungsleistung (MW)"},
            {"name": "landcover_center_class", "type": "esriFieldTypeString", "alias": "Landbedeckung am Mittelpunkt", "length": 64},
            {"name": "country", "type": "esriFieldTypeString", "alias": "Land", "length": 128},
        ],
        "extent": {
            "xmin": -180, "ymin": -90, "xmax": 180, "ymax": 90,
            "spatialReference": {"wkid": 4326},
        },
        "timeInfo": {
            "startTimeField": "acq_time",
            "timeReference": {"timeZone": "UTC", "respectsDaylightSaving": False},
            "timeInterval": 1, "timeIntervalUnits": "esriTimeUnitsHours",
        },
        "indexes": [
            {"name": "ux_detection_id", "fields": "detection_id", "isAscending": True, "isUnique": True},
            {"name": "ix_acq_time", "fields": "acq_time", "isAscending": True, "isUnique": False},
        ],
    }
    service = FeatureLayerCollection.fromitem(item)
    result = service.manager.add_to_definition({"layers": [definition]})
    if not result.get("success"):
        raise RuntimeError(f"Layer konnte nicht angelegt werden: {result}")
    item.update(item_properties={
        "title": SERVICE_TITLE,
        "snippet": "Aktuelle VIIRS-Hotspots auf Vegetation, gefiltert mit ESA WorldCover 2021.",
        "tags": [SERVICE_TAG, "VIIRS", "WorldCover"],
    })
    item = gis.content.get(item.id)

layer = item.layers[0]
print(f"Ziellayer: {item.title} ({item.id})")

## 3 · Aktuelle VIIRS-Punkte laden

Zuerst ermitteln wir das kleinste `hours_old` der Quelle. Anschließend kombiniert
die SQL-Abfrage drei Bedingungen mit `AND`: **aktueller Altersblock**, **passende
Confidence** und **Aufnahme innerhalb des Aufbewahrungsfensters**.
Die ArcGIS API lädt mit `return_all_records=True` alle passenden Ergebnisse,
auch wenn der Service sie auf mehrere Seiten verteilt.

Der erste Lauf lädt nur diesen aktuellen Ausschnitt. Der Sieben-Tage-Bestand
baut sich durch regelmäßige Läufe auf; es wird kein historischer Vollbestand geladen.

In [ ]:
statistics = source.query(
    out_statistics=[{
        "statisticType": "min", "onStatisticField": "hours_old",
        "outStatisticFieldName": "min_age",
    }],
    return_geometry=False,
)
min_age = statistics.features[0].attributes["min_age"]
confidence_sql = ", ".join(f"'{value}'" for value in CONFIDENCE_VALUES)

# Bei einer leeren Quelle liefert 1=0 eine leere Ergebnismenge.
where = "1=0"
if min_age is not None:
    max_age = int(min_age) + SOURCE_OVERLAP_HOURS
    where = (
        f"hours_old <= {max_age} AND confidence IN ({confidence_sql}) "
        f"AND acq_time >= {cutoff_sql}"
    )

print(f"VIIRS-Filter: {where}")
result = source.query(
    where=where,
    out_fields="satellite,acq_time,frp",
    out_sr=4326,
    return_all_records=True,
)

rows = []
for feature in result.features:
    attributes, geometry = feature.attributes, feature.geometry
    lon, lat = geometry["x"], geometry["y"]
    acq_time = int(attributes["acq_time"])
    satellite = attributes["satellite"] or ""
    # Die Quell-OBJECTID kann wechseln; diese Kennung bleibt stabil.
    key = f"{satellite}|{acq_time}|{lon:.5f}|{lat:.5f}"
    rows.append([key, lon, lat, acq_time, attributes["frp"]])

hotspots = pd.DataFrame(rows, columns=[
    "detection_id", "longitude", "latitude", "acq_time_ms", "frp",
]).drop_duplicates("detection_id").reset_index(drop=True)
downloaded_count = len(hotspots)
print(f"Geladene eindeutige Beobachtungen: {downloaded_count:,}")

## 4 · Bereits gespeicherte Beobachtungen überspringen

Eine Überlappung beim Abruf ist gewollt. Über `detection_id` entfernen wir die
bereits gespeicherten Punkte, bevor die Rasterabfrage beginnt. Verschiedene
Satellitenüberflüge bleiben eigenständige Beobachtungen.

In [ ]:
new = hotspots.copy()
if not new.empty:
    earliest = pd.to_datetime(new["acq_time_ms"].min(), unit="ms", utc=True)
    existing = layer.query(
        where=f"acq_time >= TIMESTAMP '{earliest:%Y-%m-%d %H:%M:%S}'",
        out_fields="detection_id",
        return_geometry=False,
        return_all_records=True,
    )
    known_ids = {feature.attributes["detection_id"] for feature in existing.features}
    new = new.loc[~new["detection_id"].isin(known_ids)].reset_index(drop=True)

new_count = len(new)
print(f"Davon noch nicht gespeichert: {new_count:,}")

## 5 · Nur Hotspots auf Vegetation behalten

ArcPy liest für jeden neuen Punkt die WorldCover-Klasse am Mittelpunkt aus.
`NEAREST` erhält die Klassenwerte: Zwischen zwei Landbedeckungen wäre ein
interpolierter Zahlenwert bedeutungslos.

Die Filterentscheidung selbst ist eine Zeile:

```python
keep = sampled["landcover_code"].isin(VEGETATION_CODES)
```

**Zum Nachvollziehen – erfundene Beispielpunkte, keine Live-Daten:**

In [ ]:
example = pd.DataFrame({
    "Beispielpunkt": ["A", "B", "C", "D", "E"],
    "Landbedeckung": ["Bäume", "Bebaute Fläche", "Ackerland", "Wasser", "Kein Rasterwert"],
    "WorldCover-Code": [10, 50, 40, 80, None],
})
example["Behalten"] = example["WorldCover-Code"].isin(VEGETATION_CODES)
display(example)

In [ ]:
def sample_worldcover(points_df):
    """Die WorldCover-Klasse am Mittelpunkt jedes Hotspots ergänzen."""
    sampled = points_df.copy()
    sampled["landcover_code"] = pd.Series(index=sampled.index, dtype="float64")
    if sampled.empty:
        return sampled

    points = r"memory\viirs_points"
    samples = r"memory\worldcover_samples"
    arcpy.management.CreateFeatureclass("memory", "viirs_points", "POINT", spatial_reference=4326)
    arcpy.management.AddField(points, "row_id", "LONG")

    # row_id verbindet jeden Rasterwert wieder mit der richtigen Tabellenzeile.
    with arcpy.da.InsertCursor(points, ["SHAPE@XY", "row_id"]) as cursor:
        for row_id, row in sampled.iterrows():
            cursor.insertRow(((row.longitude, row.latitude), row_id))

    arcpy.sa.Sample(WORLDCOVER_URL, points, samples, "NEAREST", "row_id")
    # Beim verwendeten einbandigen Raster ist das letzte Feld der Klassenwert.
    value_field = arcpy.ListFields(samples)[-1].name
    with arcpy.da.SearchCursor(samples, ["row_id", value_field]) as cursor:
        codes = dict(cursor)

    sampled["landcover_code"] = sampled.index.map(codes)
    arcpy.management.Delete(points)
    arcpy.management.Delete(samples)
    return sampled


sampled = sample_worldcover(new)
keep = sampled["landcover_code"].isin(VEGETATION_CODES)
filtered = sampled.loc[keep].copy()
filtered["landcover_center_class"] = filtered["landcover_code"].map(WORLDCOVER_CLASS)
print(f"Auf Vegetation: {len(filtered):,} von {new_count:,} neuen Beobachtungen")

## 6 · Das Land ergänzen

Ein räumlicher Join ordnet die behaltenen Punkte den Länderpolygonen zu.
`SOVEREIGNT` ist die Länderbezeichnung aus dem verwendeten Natural-Earth-Layer.
Punkte ohne Treffer bleiben erhalten und bekommen einen leeren Ländernamen.
Bei mehreren Grenztreffern verwenden wir den alphabetisch ersten Namen,
damit jede Beobachtung genau einmal gespeichert wird.

In [ ]:
filtered["country"] = ""
if not filtered.empty:
    country_df = countries.query(
        out_fields="SOVEREIGNT", out_sr=4326, return_all_records=True,
    ).sdf
    points_df = pd.DataFrame.spatial.from_xy(
        filtered.copy(), x_column="longitude", y_column="latitude", sr=4326,
    )
    joined = points_df.spatial.join(
        country_df[["SOVEREIGNT", "SHAPE"]], how="left", op="intersects",
    )
    country_by_id = (
        joined.sort_values("SOVEREIGNT")
        .drop_duplicates("detection_id")
        .set_index("detection_id")["SOVEREIGNT"]
    )
    # Über die Kennung zuordnen: Ein Join kann Reihenfolge und Zeilenzahl ändern.
    filtered["country"] = filtered["detection_id"].map(country_by_id).fillna("")

display(filtered[["country", "landcover_center_class", "frp"]].head(10))

## 7 · Den Hosted Feature Layer aktualisieren

Jetzt werden die gefilterten Punkte in Paketen von 200 gespeichert. Danach
entfernen wir alte Beobachtungen und berechnen `hours_old` für den gesamten
verbleibenden Bestand neu. Dadurch kann ein Dashboard nach dem aktuellen Alter
filtern. Die Altersangabe gilt jeweils zum letzten Notebook-Lauf.

Die kurzen Prüfungen der Schreibantworten bleiben erhalten, damit ein fehlgeschlagener
Upload oder eine fehlgeschlagene Aktualisierung nicht als Erfolg angezeigt wird.
Für einen erneuten Durchlauf wieder bei Schritt 1 starten, damit Zeit und Dubletten aktuell sind.

In [ ]:
features = []
for row in filtered.itertuples():
    features.append({
        "geometry": {
            "x": row.longitude, "y": row.latitude,
            "spatialReference": {"wkid": 4326},
        },
        "attributes": {
            "detection_id": row.detection_id,
            "acq_time": int(row.acq_time_ms),
            "hours_old": round((now_ms - row.acq_time_ms) / 3_600_000, 1),
            "frp": None if pd.isna(row.frp) else float(row.frp),
            "landcover_center_class": row.landcover_center_class,
            "country": row.country,
        },
    })

for start in range(0, len(features), 200):
    batch = features[start:start + 200]
    result = layer.edit_features(adds=batch, rollback_on_failure=True)
    saved = result.get("addResults", [])
    if len(saved) != len(batch) or not all(entry["success"] for entry in saved):
        raise RuntimeError(f"Speichern fehlgeschlagen: {result}")

# Nur Punkte außerhalb des rollenden Zeitfensters entfernen.
result = layer.delete_features(
    where=f"acq_time < {cutoff_sql}", return_delete_results=False,
)
if not result.get("success"):
    raise RuntimeError(f"Alte Punkte konnten nicht entfernt werden: {result}")

result = layer.calculate(
    where="1=1",
    calc_expression={
        "field": "hours_old",
        "sqlExpression": "ROUND((CURRENT_TIMESTAMP - acq_time) * 24, 1)",
    },
)
if not result.get("success"):
    raise RuntimeError(f"Alter konnte nicht aktualisiert werden: {result}")

## 8 · Das Ergebnis zeigen

Die Übersicht zeigt den Weg vom SQL-Ergebnis zum gespeicherten Vegetations-Hotspot.
Die Zeilen beziehen sich auf diesen Lauf; die Gesamtzahl im Layer umfasst auch
frühere Läufe. Über den Link lässt sich das Ergebnis direkt im Map Viewer ansehen.

In [ ]:
summary = pd.DataFrame({
    "Schritt": ["Nach Zeit- und Confidence-Filter", "Noch nicht gespeichert", "Auf Vegetation / hinzugefügt"],
    "Anzahl": [downloaded_count, new_count, len(features)],
})
display(summary)

total = layer.query(return_count_only=True)
print(f"Gesamtbestand im Layer: {total:,} Beobachtungen")
print(f"Start dieses Laufs (UTC): {now:%Y-%m-%d %H:%M}")
display(Markdown(
    f"[Layer öffnen](https://www.arcgis.com/home/item.html?id={item.id}) · "
    f"[Im Map Viewer ansehen](https://www.arcgis.com/apps/mapviewer/index.html?layers={item.id})"
))

## Zum Ausprobieren

- **Nur hohe Confidence:** `CONFIDENCE_VALUES = ["high"]` setzen.
- **Nur Bäume:** `VEGETATION_CODES = {10}` setzen.
- **Ackerland ausschließen:** `40` aus `VEGETATION_CODES` entfernen.
- **Drei Tage behalten:** `ROLLING_HOURS = 72` setzen.

Für vergleichbare Varianten jeweils einen neuen `SERVICE_NAME` und ein neues
`SERVICE_TAG` wählen: Bereits gespeicherte Beobachtungen werden nicht erneut
gefiltert. Die Beispieltabelle in Schritt 5 zeigt die geänderte Vegetationsauswahl sofort.

### Regelmäßig aktualisieren

Nach einem erfolgreichen manuellen Durchlauf das Notebook in ArcGIS Online
stündlich ausführen lassen. Die Abfrage erfasst den jeweils neuesten Altersblock
plus Überlappung. Nach längeren Pausen können Lücken bleiben; für einen einmaligen
Nachlauf lässt sich `SOURCE_OVERLAP_HOURS` erhöhen, soweit die Quelle diese Daten
noch bereitstellt. [Notebook-Aufgaben planen](https://doc.arcgis.com/en/arcgis-online/create-maps/prepare-a-notebook-for-automated-execution.htm).

### Was der Filter aussagt

Geprüft wird **ein WorldCover-Pixel am Mittelpunkt**, nicht die Vegetationsfläche
im gesamten VIIRS-Pixel. WorldCover bildet das Jahr 2021 ab; heutige Landbedeckung
kann abweichen. Der Filter ist eine einfache Vorauswahl möglicher Vegetationsbrände.
Die Länderzuordnung folgt den verwendeten Natural-Earth-Grenzen.

**Quellen und Methoden:**
[VIIRS-Quelllayer](https://services9.arcgis.com/RHVPKKiFTONKtxq3/arcgis/rest/services/Satellite_VIIRS_Thermal_Hotspots_and_Fire_Activity/FeatureServer/0) ·
[ESA WorldCover](https://esa-worldcover.org/en/data-access) ·
[Natural-Earth-Länderlayer](https://services.arcgis.com/0xnwbwUttaTjns4i/ArcGIS/rest/services/Natural_Earth_quick_start/FeatureServer/43) ·
[ArcPy Sample](https://pro.arcgis.com/en/pro-app/3.5/tool-reference/spatial-analyst/sample.htm) ·
[ArcGIS API for Python](https://developers.arcgis.com/python/latest/api-reference/arcgis.features.toc.html)

WorldCover: © ESA WorldCover project 2021 / Contains modified Copernicus Sentinel
data (2021) processed by ESA WorldCover consortium.